In [ ]:
import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from IPython.display import display

from tqdm import tqdm

import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

from scipy import stats

np.set_printoptions(precision=4)

from brainpred import paths, splits
from brainpred.utils import display as vis
from brainpred.utils import intensity, metrics, ml as uml
from brainpred.features.longitudinal.whole_brain_based import FeaturesLoader

clusters_folder = paths.get_work_folder("clusters")

In [ ]:
cluster_by = ["bpf_dif"]

exclude_patients = None
reader = FeaturesLoader()
data = reader.get_df(columns=cluster_by.copy(),
                     remove_columns=False, 
                     return_patient_id=True, 
                     id_patients=exclude_patients,
                     remove_patients=True)

data_ids = data.pop('patient_id').tolist()
data_columns = data.columns.tolist()
vis.plot_data_voxplot(data, data_columns, figsize=(4,2))

print("nb_patients:", len(data_ids))
print(int(splits.get_excluded_patients()[0]) in data_ids)



# preprocessing

In [ ]:
data_no_outlayers, no_outlayers_index, outlayers_index = uml.remove_outliers_zscore(data.values, threshold=2)
data_standardized, standarizer = uml.standarize_columns(data_no_outlayers)
data_normalized, normalizer = uml.normalize_columns(data_standardized)
data_normalized_df = pd.DataFrame(data_normalized, columns=data_columns)
vis.plot_data_voxplot(data_normalized_df, data_columns, figsize=(8,4))

data_outlayers = data.values[outlayers_index]
data_outlayers = standarizer.transform(data_outlayers)
data_outlayers = normalizer.transform(data_outlayers)
data_outlayers_df = pd.DataFrame(data_outlayers, columns=data_columns)
vis.plot_data_voxplot(data_outlayers_df, data_columns, figsize=(8,4))

print("nb NO outlayers:", len(no_outlayers_index))
print("nb outlayers:", len(outlayers_index))


# Clustering

In [ ]:
cluster_with_center = False

if cluster_with_center:
    clusters = uml.cluster_by_mean_with_center_df(data_normalized_df, cluster_by, 0.2)
else :
    clusters = uml.cluster_by_mean_df(data_normalized_df, cluster_by)

print(np.unique(clusters, return_counts=True))

vis.visualize_clusters(data_normalized_df, clusters, cluster_by, figsize=(10,5), hist=False)


# Cluster outlayers

In [ ]:
data_concat = np.concatenate([data_normalized, data_outlayers], axis=0)
data_concat_df = pd.DataFrame(data_concat, columns=data_columns)

concatenate_fake_clusters = list(tuple(clusters)+ tuple([4]*len(outlayers_index)))

print(np.unique(concatenate_fake_clusters, return_counts=True))
vis.visualize_clusters(data_concat_df, concatenate_fake_clusters, cluster_by, figsize=(10,5), hist=False)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(data_normalized_df[cluster_by], clusters)
clusters_outlayers = knn.predict(data_outlayers_df[cluster_by])
concatenate_clusters = list(tuple(clusters)+ tuple(clusters_outlayers))

print(np.unique(clusters_outlayers, return_counts=True))
vis.visualize_clusters(data_concat_df, concatenate_clusters, cluster_by, figsize=(10,5), hist=False)


In [ ]:
concatenate_ids = tuple(np.array(data_ids)[no_outlayers_index]) + tuple(np.array(data_ids)[outlayers_index])
df = pd.DataFrame(zip(concatenate_ids,concatenate_clusters), columns=['patient_id', 'bpf_clusters']).astype(int)
ordered_df = df.sort_values(by='patient_id')

print(np.unique(concatenate_clusters, return_counts=True))

name = f"longitudinal_wbb-{'_'.join(cluster_by)}-{len(np.unique(concatenate_clusters))}"
ordered_df.to_csv(f"{clusters_folder}/{name}.csv", index=False)
print(ordered_df.head())

# [149, 209, 207, 138]
# [ 68,  75, 112,  59,  62,  74, 118,  74,  61]